# DSCI 619 Deep Learning Project 1
## Ben Miller

In [6]:
import pandas as pd
import os
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, root_mean_squared_error, mean_absolute_error
from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

## Load the dataset into memory

In [7]:
df = pd.read_csv("airfoil_self_noise.dat", sep="\t")
# name the columns
df.columns = ["Frequency", "Angle of Attack", "Chord Length", "Free Stream Velocity", "Suction Side Displacement Thickness", "Scaled Sound Pressure Level"]
df.head()

,Frequency,Angle of Attack,Chord Length,Free Stream Velocity,Suction Side Displacement Thickness,Scaled Sound Pressure Level
0,1000,0.0,0.3048,71.3,0.002663,125.201
1,1250,0.0,0.3048,71.3,0.002663,125.951
2,1600,0.0,0.3048,71.3,0.002663,127.591
3,2000,0.0,0.3048,71.3,0.002663,127.461
4,2500,0.0,0.3048,71.3,0.002663,125.571


## Check for missing values

In [8]:
df.isnull().sum(axis=0)

,0
Frequency,0
Angle of Attack,0
Chord Length,0
Free Stream Velocity,0
Suction Side Displacement Thickness,0
Scaled Sound Pressure Level,0


## Split the training data

In [9]:
X = df.drop("Scaled Sound Pressure Level", axis=1)
y = df["Scaled Sound Pressure Level"]

# 80% training and 20% testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state = 42)

## Build a simple linear regression to forecast "Scaled sound pressure level" using scikit-learn package

In [10]:
model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
test_error = mean_squared_error(y_test, y_pred)
print(f"Mean Squared Error of Linear Regression: {test_error:.4f}")

Mean Squared Error of Linear Regression: 21.2594


## Preprocess the data using the normalization method

In [11]:
## MinMaxScaler scales the features to a range of [0, 1]
scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

## Create a deep learning regression model with TensorFlow

In [12]:
# Helper function to build a TensorFlow model
def build_TensorFlow_model(num_neurons, optimization_algorithm):
    model = keras.Sequential()
    model.add(layers.Dense(num_neurons, activation='relu'))
    model.add(layers.Dense(1))
    
    model.compile(optimizer=optimization_algorithm,loss='mse')
    
    return model

In [13]:
# Helper function to print Error metrics
def print_evaluation_metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    rmse = root_mean_squared_error(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    
    print(f"Mean Squared Error: {mse:.4f}")
    print(f"Root Mean Squared Error: {rmse:.4f}")
    print(f"Mean Absolute Error: {mae:.4f}")

In [14]:
tf.random.set_seed(1)
# start with 20 neurons in the hidden layer and adam optimization algorithm
model = build_TensorFlow_model(num_neurons=20, optimization_algorithm='adam')

model.fit(x=X_train,y=y_train,batch_size=64,epochs=500,
          validation_data=(X_test,y_test)
          )

Epoch 1/500
19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 15625.4707 - val_loss: 15558.7197
Epoch 2/500
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 15574.9795 - val_loss: 15509.1328
Epoch 3/500
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 15521.6289 - val_loss: 15455.5947
Epoch 4/500
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 15462.9561 - val_loss: 15395.0361
Epoch 5/500
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 15395.7637 - val_loss: 15324.7061
Epoch 6/500
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 15317.5371 - val_loss: 15241.9834
Epoch 7/500
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 15225.7969 - val_loss: 15144.4033
Epoch 8/500
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 15117.5938 - val_loss: 15029.4434
Epoch 9/500
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 14990.7012 - val_loss: 14896.7012
Epoch 10/500
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 14847.2939 - val_loss: 14750.5479
Epoch 11/500
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 14691.75

In [15]:
y_pred = model.predict(X_test)
print_evaluation_metrics(y_test, y_pred)

10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step 
Mean Squared Error: 21.5678
Root Mean Squared Error: 4.6441
Mean Absolute Error: 3.5558


## Attempt to improve the performance by adjusting the number of neurons and optimization algorithm

In [16]:
# increase the number of neurons to 50 and change the optimization algorithm to stochastic gradient descent (sgd)
improved_model = build_TensorFlow_model(num_neurons=50, optimization_algorithm='sgd')

improved_model.fit(x=X_train,y=y_train,batch_size=64,epochs=500,
          validation_data=(X_test,y_test)
          )

Epoch 1/500
19/19 ━━━━━━━━━━━━━━━━━━━━ 1s 12ms/step - loss: 10885.2705 - val_loss: 9853.6846
Epoch 2/500
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 7288.8242 - val_loss: 5079.7915
Epoch 3/500
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 3662.7988 - val_loss: 2379.3455
Epoch 4/500
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 1726.0380 - val_loss: 1127.4414
Epoch 5/500
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 826.9630 - val_loss: 547.3548
Epoch 6/500
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 409.5402 - val_loss: 278.7581
Epoch 7/500
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 215.6995 - val_loss: 154.5230
Epoch 8/500
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 125.6577 - val_loss: 97.1510
Epoch 9/500
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 83.8140 - val_loss: 70.7187
Epoch 10/500
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 64.3560 - val_loss: 58.5831
Epoch 11/500
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 9ms/step - loss: 55.2993 - val_loss: 53.0406
Epoch 12/500
19/

In [17]:
y_pred = improved_model.predict(X_test)
print_evaluation_metrics(y_test, y_pred)

10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step 
Mean Squared Error: 48.5975
Root Mean Squared Error: 6.9712
Mean Absolute Error: 5.7169


## Create a deep learning regression model with PyTorch

In [18]:
X_train = torch.tensor(X_train.astype(np.float32))
y_train = torch.tensor(y_train.values.astype(np.float32).reshape(-1,1))

In [19]:
# define the child module class derivated from parent class of torch.nn.Module)
class LinearRegressionModel(torch.nn.Module):
    #define the constructor
    def __init__(self, input_size, hidden_size, output_size):
        super(LinearRegressionModel, self).__init__()
        # first layer: input -> hidden
        self.hidden = torch.nn.Linear(input_size, hidden_size) 
        # second layer: hidden -> output
        self.predict = torch.nn.Linear(hidden_size, output_size) 
    #override the forward function in this child class
    def forward(self, x):
        x = F.relu(self.hidden(x))     
        y_pred = self.predict(x)            
        return y_pred

In [20]:
def train_PyTorch_model(model, l, optimizer, X_train, y_train):
  #fix the random seeds for torch and np
  torch.manual_seed(1)
  np.random.seed(0)

  #set the number of epochs
  num_epochs = 500
  for epoch in range(num_epochs):
      #forward pass
      y_pred = model(X_train.requires_grad_())

      #calculate the loss
      loss= l(y_pred, y_train)
      #Set the gradients to be zero
      optimizer.zero_grad()
    
      #backward pass: calculate gradients
      loss.backward()

      #update the weights
      optimizer.step()
  
      print('epoch {0}, loss:{1:.4f}'.format(epoch, loss.item()))

In [21]:
# start with 20 neurons in the hidden layer and adam optimization algorithm
model = LinearRegressionModel(input_size = X_train.shape[1], hidden_size = 20, output_size = 1)
loss = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.05)
train_PyTorch_model(model, loss, optimizer, X_train, y_train)

epoch 0, loss:15679.5957
epoch 1, loss:15601.0986
epoch 2, loss:15513.7920
epoch 3, loss:15408.8145
epoch 4, loss:15281.1445
epoch 5, loss:15127.9814
epoch 6, loss:14947.5635
epoch 7, loss:14738.5166
epoch 8, loss:14499.6768
epoch 9, loss:14230.3281
epoch 10, loss:13929.9004
epoch 11, loss:13597.8115
epoch 12, loss:13233.3555
epoch 13, loss:12835.9365
epoch 14, loss:12404.9775
epoch 15, loss:11942.4785
epoch 16, loss:11449.5703
epoch 17, loss:10927.8301
epoch 18, loss:10378.7969
epoch 19, loss:9804.3965
epoch 20, loss:9207.1123
epoch 21, loss:8589.9521
epoch 22, loss:7956.6128
epoch 23, loss:7312.7021
epoch 24, loss:6664.5430
epoch 25, loss:6018.3530
epoch 26, loss:5380.7710
epoch 27, loss:4758.9409
epoch 28, loss:4160.4072
epoch 29, loss:3592.9763
epoch 30, loss:3064.5291
epoch 31, loss:2582.7698
epoch 32, loss:2154.9045
epoch 33, loss:1787.2452
epoch 34, loss:1484.7321
epoch 35, loss:1250.4006
epoch 36, loss:1084.8174
epoch 37, loss:985.5709
epoch 38, loss:946.9129
epoch 39, loss:959

In [22]:
#convert numpy to tensor
X_test_tensor = torch.from_numpy(X_test.astype(np.float32))
#Stop tracking the gradient by calling detach since we don't use it anymore
y_pred = model(X_test_tensor).detach().numpy()

print_evaluation_metrics(y_test, y_pred)

Mean Squared Error: 22.0893
Root Mean Squared Error: 4.6999
Mean Absolute Error: 3.6132


Attempt to improve the performance by adjusting the number of neurons and optimization algorithm

In [23]:
# increase the number of neurons to 50
improved_model = LinearRegressionModel(input_size = X_train.shape[1], hidden_size = 50, output_size = 1)
loss = nn.MSELoss()
# change the optimization algorithm to stochastic gradient descent (sgd) with a learning rate of 0.001
optimizer = torch.optim.SGD(improved_model.parameters(), lr=0.001)
train_PyTorch_model(improved_model, loss, optimizer, X_train, y_train)

epoch 0, loss:15625.0342
epoch 1, loss:15392.9805
epoch 2, loss:15133.4404
epoch 3, loss:14784.1387
epoch 4, loss:14268.7393
epoch 5, loss:13483.0684
epoch 6, loss:12293.9219
epoch 7, loss:10570.8906
epoch 8, loss:8279.9658
epoch 9, loss:5640.2803
epoch 10, loss:3194.5847
epoch 11, loss:1519.7438
epoch 12, loss:726.6826
epoch 13, loss:466.9098
epoch 14, loss:399.0886
epoch 15, loss:378.0785
epoch 16, loss:366.2358
epoch 17, loss:356.2216
epoch 18, loss:346.7849
epoch 19, loss:337.7145
epoch 20, loss:328.9665
epoch 21, loss:320.5220
epoch 22, loss:312.3680
epoch 23, loss:304.4923
epoch 24, loss:296.8830
epoch 25, loss:289.5294
epoch 26, loss:282.4214
epoch 27, loss:275.5488
epoch 28, loss:268.9025
epoch 29, loss:262.4736
epoch 30, loss:256.2535
epoch 31, loss:250.2342
epoch 32, loss:244.4081
epoch 33, loss:238.7679
epoch 34, loss:233.3063
epoch 35, loss:228.0169
epoch 36, loss:222.8933
epoch 37, loss:217.9293
epoch 38, loss:213.1190
epoch 39, loss:208.4570
epoch 40, loss:203.9378
epoch 

In [24]:
#convert numpy to tensor
X_test_tensor = torch.from_numpy(X_test.astype(np.float32))
#Stop tracking the gradient by calling detach since we don't use it anymore
y_pred = improved_model(X_test_tensor).detach().numpy()

print_evaluation_metrics(y_test, y_pred)

Mean Squared Error: 23.1341
Root Mean Squared Error: 4.8098
Mean Absolute Error: 3.7710
